En este notebook vamos a crear un clasificador Naive Bayes para resolver un problema de Sentiment Analysis de Twitts. Este problema se enmarca dentro de NLU (Natural Language Understanding) ya que nos permitirá explotar los datos de tweets para extraer información de los mismos.
Por Prof Victoria Gutiérrez

# 1. Importar bibliotecas necesarias

In [32]:
import pandas as pd

# Bibliotecas para preprocesamiento y vectorización

from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
import re

#Bibliotecas para entrenar el modelo
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 2. Analizamos el dataset

In [45]:
file_path = "training.1600000.processed.noemoticon.csv"

df = pd.read_csv(
    file_path,
    encoding="latin-1",
    header=None,
    names=["target", "id", "date", "flag", "user", "text"],
    sep=",",             # correct separator
    quotechar='"',       # handles commas inside tweets
    engine="python",      # more flexible for irregular lines
    skiprows=1
)

print(df.head())
print(df.shape)

   target          id                          date      flag        user  \
0       0  1998864136  Mon Jun 01 19:15:00 PDT 2009  NO_QUERY   stratagee   
1       0  1998864367  Mon Jun 01 19:15:01 PDT 2009  NO_QUERY  hbturner75   
2       0  1998864497  Mon Jun 01 19:15:02 PDT 2009  NO_QUERY   sher_in_a   
3       0  1998864564  Mon Jun 01 19:15:02 PDT 2009  NO_QUERY  move_a1ong   
4       0  1998864573  Mon Jun 01 19:15:02 PDT 2009  NO_QUERY    Nelly_Vi   

                                                text  
0  Vanished one day and I really hope he is oook ...  
1               Wanting to be with my man but can't   
2                        missing my sweet lily Ava!   
3                   still sick  school tomorroww UGH  
4                    Off work And feelin like shit.   
(1297884, 6)


In [46]:
df.tail()

,target,id,date,flag,user,text
1297879,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1297880,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1297881,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1297882,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1297883,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [47]:
df['target'].value_counts()

,count
target,
4,800000
0,497884


**¿Es estructurado, semi-estructurado o no estructurado?**

**¿Cuantas observaciones tengo?**

**¿Qué variables voy a usar para entrenar el modelo de sentiment analysis?**

**¿En qué idioma está el texto? ¿Que limpieza y preprocesamiento será necesario llevar a cabo?**

**¿Qué valores toma la variable de salida? ¿Qué significa cada valor? ¿Las clases están balanceadas?**

# 3. Limpieza y Preprocesamiento

a. Descargamos stopwords en inglés

In [48]:
nltk.download('stopwords')
stop_words = list(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


b. Definimos una función para limpiar el texto. Quitamos términos usando regex (regular expression)

In [49]:
def clean_text(text):
    if text is None:  # Handle None values
        return ""
    text = re.sub(r'@\w+', '', text)  # Eliminar menciones de usuarios
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Eliminar URLs
    text = re.sub(r'\d+', '', text)  # Eliminar números
    text = re.sub(r'[^\w\s]', '', text)  # Eliminar caracteres especiales y puntuación
    text = re.sub(r'\b\w*[ñð¾ºµ¼]\w*\b', '', text)  # Eliminar palabras que contienen los caracteres especiales
    text = text.lower()  # Convertir a minúsculas
    return text

3. Nos quedamos solo con las columnas necesarias y limpiamos el texto

In [50]:
df = df[['target', 'text']]
df['target'] = df['target'].map({0: 'negativo', 4: 'positivo'})

df['text'] = df['text'].apply(clean_text)

4. Hacemos una muestra para que sea más eficiente el entrenamiento

In [52]:
df_sample = df.sample(n=50000, random_state=42)

df_sample

,target,text
1219130,positivo,fun day screams and laughter
891304,positivo,great song thanks â
313867,negativo,missing and the rest of the fam already its r...
1045944,positivo,i guess it isnt very different out there its ...
1029496,positivo,oh i was fine love i had to go up north for a...
...,...,...
194416,negativo,great time at driving range but now see suppos...
113005,negativo,why never reply all my messages
760203,positivo,not to worry noone got that one next question ...
1050060,positivo,cant wait for next weekend when eric comes back


**Chequear que las clases estén balanceadas**

5. Separamos X e y

In [53]:
texts = df_sample['text'].tolist()
labels = df_sample['target'].tolist()

5. Vectorizamos textos y quitamos stopwords (se hace todo en un comando)

In [57]:
?CountVectorizer

**¿Qué hace CountVectorizer?**

In [58]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(texts)
X_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

X_df


,__,___,____,_____,___quotim,__x,_amplt,_bam_,_flying,_je,...,ðññð²ðñññ,ñðð³ðññ,ñððð²ðññ,ñððñð½ñð¹,ñðñðð²ððññ,ñðñðñññ,ñðñññðððñ,ññðññññ,ññññðð½ñð,ùùùùùùùùù
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**¿Cuántas variables tiene el dataset? ¿Ya está listo para entrenar el modelo?**

# 4. Entrenamiento del Modelo

Ya tenemos los datos separados en X e y (se llaman 'X' y 'labels')

**En 5 líneas de código, entrenar un modelo de Naive Bayes (MultinomialNB()) y calcular accuracy en test**

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.25, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.75


# 5. Análisis de Resultados

**¿Están conformes con la performance del modelo? ¿Qué se puede hacer para mejorarla?**


**Analicen las palabras más frecuentes dentro de cada clase ¿Tienen sentido?**


**¿Ven alguna manera de mejorar el preprocesamiento?**

In [60]:
# Create sample tweets
sample_tweets = [
    "This is a terrible day, I'm so sad.", # Negative
    "I hate this weather, it's awful.",    # Negative
    "What a beautiful day, I'm feeling great!" # Positive
]

# Clean and vectorize the sample tweets
cleaned_sample_tweets = [clean_text(tweet) for tweet in sample_tweets]
vectorized_sample_tweets = vectorizer.transform(cleaned_sample_tweets)

# Predict sentiment
predictions = model.predict(vectorized_sample_tweets)

# Display the predictions
for tweet, prediction in zip(sample_tweets, predictions):
    print(f"Tweet: '{tweet}' -> Predicted Sentiment: {prediction}")

Tweet: 'This is a terrible day, I'm so sad.' -> Predicted Sentiment: negativo
Tweet: 'I hate this weather, it's awful.' -> Predicted Sentiment: negativo
Tweet: 'What a beautiful day, I'm feeling great!' -> Predicted Sentiment: positivo
